# Workspace Building

This notebook demonstrates how to build a serialized workspace from a YAML fit configuration file.
The workspace is a JSON dictionary (following a pyhf-like schema) that contains all the information
needed to construct the statistical model: binned histograms, density ratios, systematic variations,
normalization factors, and measurement definitions.

Once built, the workspace is saved to disk as JSON and can be loaded independently for fitting
in `parameter_fitting.ipynb` — no config file or ROOT files needed at fit time.

In [1]:
import os
import pprint
import nsbi_common_utils

## Configuration

Point to your fit configuration YAML files. These define samples, regions, systematics,
and paths to ROOT files and density ratio `.npy` files.

In [2]:
hist_config_path = "./config_fit_histogram.yml"
nsbi_config_path = "./config_fit_nsbi.yml"

output_dir = "./workspaces"
os.makedirs(output_dir, exist_ok=True)

## Observed dataset

The fit needs an observed dataset: the sample flagged `Data: True` in the config. This example has no real data, so the observation is the Asimov dataset, i.e. every nominal MC event with its MC weight, written once to the file the config points to. The fit then recovers the injected parameter values. With real data, point `SamplePath` at the data file instead and skip this cell.

In [3]:
import uproot
import awkward as ak
from nsbi_common_utils.configuration import ConfigManager

samples = ConfigManager(file_path_string=nsbi_config_path).config["Samples"]
obs_sample = next(s for s in samples if s.get("Data", False))
mc_samples = [s for s in samples if not s.get("Data", False)]

if os.path.exists(obs_sample["SamplePath"]):
    print(f"Observed dataset already exists: {obs_sample['SamplePath']}")
else:
    # Streamed in chunks so the full dataset is never held in memory.
    with uproot.recreate(obs_sample["SamplePath"]) as fout:
        tree = None
        for s in mc_samples:
            for chunk in uproot.iterate(f"{s['SamplePath']}:{s['Tree']}", step_size="500 MB", library="ak"):
                columns = {name: ak.to_numpy(chunk[name]) for name in ak.fields(chunk) if name != "sample_name"}
                columns["sample_name"] = ak.Array([obs_sample["Name"]] * len(chunk))
                if tree is None:
                    tree = fout.mktree(obs_sample["Tree"], {name: ("string" if name == "sample_name" else arr.dtype) for name, arr in columns.items()})
                tree.extend(columns)
    print(f"Wrote {obs_sample['SamplePath']}")


Wrote ./saved_datasets/dataset_with_obs.root


## Build histogram workspace

The histogram workspace contains only binned channels (control regions, binned signal region).
This is the traditional HistFactory-style analysis.

In [4]:
builder_hist = nsbi_common_utils.workspace_builder.WorkspaceBuilder(config_path=hist_config_path)
ws_hist = builder_hist.build()
builder_hist.dump_workspace(ws_hist, os.path.join(output_dir, "workspace_histogram.json"))

Wrote ./workspaces/workspace_histogram.json


## Build NSBI workspace

The NSBI workspace extends the histogram workspace with unbinned channels that use
neural density ratios. It references pre-computed `.npy` ratio files and Asimov weights.

In [5]:
builder_nsbi = nsbi_common_utils.workspace_builder.WorkspaceBuilder(config_path=nsbi_config_path)
ws_nsbi = builder_nsbi.build()
builder_nsbi.dump_workspace(ws_nsbi, os.path.join(output_dir, "workspace_nsbi.json"))

Wrote ./workspaces/workspace_nsbi.json


## Inspect the workspace

The workspace is a dictionary with `channels`, `observations`, `measurements`, and `version` keys.
Each channel contains samples with nominal data and modifiers (normfactors, systematics).
Each observation holds the observed histogram of a region; for unbinned channels it also
points to the per-event weights and the density ratio arrays evaluated on the observed events.

In [6]:
print("NSBI workspace keys:", list(ws_nsbi.keys()))
print(f"\nNumber of channels: {len(ws_nsbi['channels'])}")
for ch in ws_nsbi['channels']:
    print(f"  - {ch['name']} (type: {ch.get('type', 'binned')}, samples: {[s['name'] for s in ch['samples']]})")
print("\nObservations:")
for obs in ws_nsbi['observations']:
    print(f"  - {obs['name']}: data={obs['data']}" + (f", weights={obs['weights']}, ratios for {list(obs['ratios'])}" if 'ratios' in obs else ""))
print(f"\nMeasurement: {ws_nsbi['measurements'][0]['name']}")
print(f"POI: {ws_nsbi['measurements'][0]['config']['poi']}")
print(f"Parameters: {[p['name'] for p in ws_nsbi['measurements'][0]['config']['parameters']]}")

NSBI workspace keys: ['measurements', 'channels', 'observations', 'version']

Number of channels: 3
  - CR (type: binned, samples: ['ztautau', 'ttbar', 'htautau'])
  - SR_binned (type: binned, samples: ['ztautau', 'ttbar', 'htautau'])
  - SR (type: unbinned, samples: ['ztautau', 'ttbar', 'htautau'])

Observations:
  - CR: data=[15320.3046875, 84141.6796875, 612157.5625, 199734.0625, 84123.875]
  - SR_binned: data=[21.231264114379883]
  - SR: data=[50268.3515625], weights=./saved_datasets/asimov_weights.npy, ratios for ['htautau', 'ttbar', 'ztautau']

Measurement: higgs_tautau_signal_strength
POI: mu_htautau
Parameters: ['mu_htautau', 'mu_ztautau', 'mu_ttbar', 'JES', 'TES']


In [7]:
print("Full NSBI workspace:\n")
pprint.pprint(ws_nsbi)

Full NSBI workspace:

{'channels': [{'name': 'CR',
               'samples': [{'data': [475.87244,
                                     67731.52,
                                     605363.5,
                                     197165.69,
                                     82408.81],
                            'modifiers': [{'data': None,
                                           'name': 'mu_ztautau',
                                           'type': 'normfactor'},
                                          {'data': {'hi_data': [0.99743,
                                                                0.9957975,
                                                                0.99925584,
                                                                1.0009964,
                                                                1.0035554],
                                                    'lo_data': [1.0007852,
                                                                1.0041897

## Next steps

The saved workspace JSON files can now be loaded in `parameter_fitting.ipynb` for
model construction and hypothesis testing — without needing the original config files
or ROOT files at fit time.

```python
ws = nsbi_common_utils.workspace_builder.WorkspaceBuilder.load_workspace("workspaces/workspace_nsbi.json")
model = nsbi_common_utils.models.sbi_parametric_model(workspace=ws, measurement_to_fit="my_measurement")
```